# HistGBM hyperparameter tuning

In [1]:
# Load data

import pandas as pd
import numpy as np 

train_df = pd.read_parquet("data/train_data.parquet")

train_df.index = pd.to_numeric(train_df.index, errors="coerce")

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop, errors="ignore")

In [2]:
# Separate features and targets

train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"]
train_full_y_reg = train_df["target_annual_roi"]

# Drop datetime features from the feature set

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_full_X = train_full_X.drop(columns=cols_to_drop, errors="ignore")

In [3]:
# Create subsets of the data for different training sizes - chronological order is preserved, so we take the last N rows for each subset

train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

In [21]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

cols_to_nominal_cat = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5

cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()

ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

ohe_transformer = OneHotEncoder(
    categories=ohe_categories, 
    drop="if_binary", 
    handle_unknown="ignore", 
    sparse_output=False
)

target_transformer_nominal = TargetEncoder(target_type="binary", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te)
    ],
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

imputed_numeric_preprocessor = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="median")
).set_output(transform="pandas")

gbdt_preprocessor = "passthrough"

Categorical columns:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


## Classification

[Parameters](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html)

### 1k

In [5]:
# Check the dates of 1k subset to ensure all data is from same month

print(train_1k_X["issue_d_month"].unique())
print(train_1k_X["issue_d_year"].unique())

<IntegerArray>
[10]
Length: 1, dtype: Int64
<IntegerArray>
[2016]
Length: 1, dtype: Int64


In [6]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_cat.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_cat.head(200)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
        "class_weight": "balanced", # Adjust weights inversely proportional to class frequencies
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_1k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_1k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_1k_parallel.html")


[I 2026-04-26 21:21:50,444] A new study created in memory with name: no-name-01d8c792-a99b-422b-9c22-57c5a13f0e49
[I 2026-04-26 21:21:59,839] Trial 4 finished with value: 0.6461440666613081 and parameters: {'max_iter': 138, 'learning_rate': 0.049870691033932206, 'max_depth': 4, 'min_samples_leaf': 18, 'max_features': 0.9086060460709571, 'max_leaf_nodes': 30, 'l2_regularization': 5.805278448208799e-05, 'max_bins': 78}. Best is trial 4 with value: 0.6461440666613081.
[I 2026-04-26 21:22:01,923] Trial 2 finished with value: 0.6527295501433432 and parameters: {'max_iter': 477, 'learning_rate': 0.04994333699473511, 'max_depth': 2, 'min_samples_leaf': 88, 'max_features': 0.9882621103063303, 'max_leaf_nodes': 11, 'l2_regularization': 41.21978757651541, 'max_bins': 161}. Best is trial 2 with value: 0.6527295501433432.
[I 2026-04-26 21:22:02,979] Trial 7 finished with value: 0.6482966745897779 and parameters: {'max_iter': 446, 'learning_rate': 0.03238343433314104, 'max_depth': 4, 'min_samples_l


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 21:27:48,952] Trial 116 finished with value: 0.6588792337930269 and parameters: {'max_iter': 457, 'learning_rate': 0.017558452250410073, 'max_depth': 5, 'min_samples_leaf': 62, 'max_features': 0.7268745743550109, 'max_leaf_nodes': 28, 'l2_regularization': 11.20747082000102, 'max_bins': 211}. Best is trial 17 with value: 0.6835827486689556.
[I 2026-04-26 21:27:49,469] Trial 120 finished with value: 0.663641951917814 and parameters: {'max_iter': 393, 'learning_rate': 0.01833203147132201, 'max_depth': 2, 'min_samples_leaf': 75, 'max_features': 0.4219659481783551, 'max_leaf_nodes': 25, 'l2_regularization': 9.567561319684136, 'max_bins': 239}. Best is trial 17 with value: 0.6835827486689556.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 21:27:50,820] Trial 118 finished with value: 0.6426708827570896 and parameters: {'max_iter': 393, 'learning_rate': 0.017173761303421, 'max_depth': 4, 'min_samples_leaf': 64, 'max_features': 0.42936263941813246, 'max_leaf_nodes': 25, 'l2_regularization': 11.96589348210811, 'max_bins': 239}. Best is trial 17 with value: 0.6835827486689556.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 21:27:51,940] Trial 114 finished with value: 0.6711261248330214 and parameters: {'max_iter': 726, 'learning_rate': 0.017546885034766264, 'max_depth': 6, 'min_samples_leaf': 62, 'max_features': 0.4244183349061808, 'max_leaf_nodes': 28, 'l2_regularization': 2.3906486152787307, 'max_bins': 250}. Best is trial 17 with value: 0.6835827486689556.
[I 2026-04-26 21:27:52,376] Trial 121 finished with value: 0.6646316805799565 and parameters: {'max_iter': 461, 'learning_rate': 0.017482226678186576, 'max_depth': 4, 'min_samples_leaf': 75, 'max_features': 0.889805615207032, 'max_leaf_nodes': 24, 'l2_regularization': 13.125528135891583, 'max_bins': 241}. Best is trial 17 with value: 0.6835827486689556.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 21:27:52,722] Trial 122 finished with value: 0.6530312503588367 and parameters: {'max_iter': 390, 'learning_rate': 0.0053178818847294095, 'max_depth': 4, 'min_samples_leaf': 83, 'max_features': 0.42592992726129536, 'max_leaf_nodes': 29, 'l2_regularization': 0.750811280040004, 'max_bins': 242}. Best is trial 17 with value: 0.6835827486689556.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.6836
BEST PARAMETERS:
best_params = {
    "max_iter": 809,
    "learning_rate": 0.005855268047892045,
    "max_depth": 6,
    "min_samples_leaf": 73,
    "max_features": 0.6326799701936273,
    "max_leaf_nodes": 31,
    "l2_regularization": 0.00047402616662400567,
    "max_bins": 252,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.5209
  min_samples_leaf    : 0.2118
  max_iter            : 0.1311
  max_bins            : 0.0395
  max_leaf_nodes      : 0.0345
  max_depth           : 0.0318
  max_features        : 0.0279
  l2_regularization   : 0.0025


In [7]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_histgbm.best_params.copy()
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")


final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingClassifier(**best_params)
)

final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print(f"Optuna Val AUC: {study_histgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")

BEST PARAMS: {'max_iter': 809, 'learning_rate': 0.005855268047892045, 'max_depth': 6, 'min_samples_leaf': 73, 'max_features': 0.6326799701936273, 'max_leaf_nodes': 31, 'l2_regularization': 0.00047402616662400567, 'max_bins': 252, 'random_state': 42, 'verbose': 0}
Optuna Val AUC: 0.6836
Holdout Test AUC: 0.6618


### 10k

In [8]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_cat.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_cat.tail(2000)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
        "class_weight": "balanced", # Adjust weights inversely proportional to class frequencies
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_10k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_10k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_10k_parallel.html")

[I 2026-04-26 21:28:03,314] A new study created in memory with name: no-name-c1f1120d-d6d5-42b8-a41d-be3c924a1a6e
[I 2026-04-26 21:28:28,943] Trial 5 finished with value: 0.6842921617403864 and parameters: {'max_iter': 536, 'learning_rate': 0.04273729226157769, 'max_depth': 2, 'min_samples_leaf': 22, 'max_features': 0.5615864857963939, 'max_leaf_nodes': 3, 'l2_regularization': 0.0012552474998858679, 'max_bins': 145}. Best is trial 5 with value: 0.6842921617403864.
[I 2026-04-26 21:28:31,861] Trial 0 finished with value: 0.689578367161165 and parameters: {'max_iter': 431, 'learning_rate': 0.011138477135859056, 'max_depth': 3, 'min_samples_leaf': 82, 'max_features': 0.46225353256853047, 'max_leaf_nodes': 4, 'l2_regularization': 0.0028718365065986892, 'max_bins': 228}. Best is trial 0 with value: 0.689578367161165.
[I 2026-04-26 21:28:34,971] Trial 1 finished with value: 0.6862943022020623 and parameters: {'max_iter': 380, 'learning_rate': 0.024717312122542454, 'max_depth': 8, 'min_sample


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 22:18:31,056] Trial 243 finished with value: 0.6964303391716317 and parameters: {'max_iter': 570, 'learning_rate': 0.0077495085668611305, 'max_depth': 4, 'min_samples_leaf': 41, 'max_features': 0.24647085739209407, 'max_leaf_nodes': 21, 'l2_regularization': 18.751161573712167, 'max_bins': 185}. Best is trial 142 with value: 0.6985397911125586.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 22:18:35,002] Trial 241 finished with value: 0.6969782160666769 and parameters: {'max_iter': 635, 'learning_rate': 0.007076234895884112, 'max_depth': 4, 'min_samples_leaf': 41, 'max_features': 0.24574941974522507, 'max_leaf_nodes': 24, 'l2_regularization': 58.51694955414919, 'max_bins': 185}. Best is trial 142 with value: 0.6985397911125586.
[I 2026-04-26 22:18:41,416] Trial 244 finished with value: 0.6980978669372184 and parameters: {'max_iter': 566, 'learning_rate': 0.007120882907092596, 'max_depth': 4, 'min_samples_leaf': 41, 'max_features': 0.24903089868741998, 'max_leaf_nodes': 21, 'l2_regularization': 49.20488673026745, 'max_bins': 184}. Best is trial 142 with value: 0.6985397911125586.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 22:19:08,141] Trial 247 finished with value: 0.6957486559665532 and parameters: {'max_iter': 663, 'learning_rate': 0.006989865615664451, 'max_depth': 4, 'min_samples_leaf': 41, 'max_features': 0.24200445868905715, 'max_leaf_nodes': 24, 'l2_regularization': 24.398995506221773, 'max_bins': 172}. Best is trial 142 with value: 0.6985397911125586.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 22:19:08,860] Trial 248 finished with value: 0.6961361583967962 and parameters: {'max_iter': 640, 'learning_rate': 0.007001001756806957, 'max_depth': 4, 'min_samples_leaf': 40, 'max_features': 0.24579779489769915, 'max_leaf_nodes': 22, 'l2_regularization': 23.52580466589535, 'max_bins': 163}. Best is trial 142 with value: 0.6985397911125586.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.6985
BEST PARAMETERS:
best_params = {
    "max_iter": 676,
    "learning_rate": 0.009599521078188419,
    "max_depth": 4,
    "min_samples_leaf": 53,
    "max_features": 0.2772575122505353,
    "max_leaf_nodes": 26,
    "l2_regularization": 93.90576777776629,
    "max_bins": 162,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.5465
  max_iter            : 0.1994
  max_bins            : 0.0941
  l2_regularization   : 0.0554
  max_features        : 0.0536
  max_leaf_nodes      : 0.0264
  min_samples_leaf    : 0.0178
  max_depth           : 0.0068


In [9]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 2)
best_params["learning_rate"] = best_params["learning_rate"] / 2

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingClassifier(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

BEST PARAMS: {'max_iter': 1352, 'learning_rate': 0.0047997605390942095, 'max_depth': 4, 'min_samples_leaf': 53, 'max_features': 0.2772575122505353, 'max_leaf_nodes': 26, 'l2_regularization': 93.90576777776629, 'max_bins': 162, 'random_state': 42, 'verbose': 0}

Optuna Val AUC:   0.6985
Holdout Test AUC: 0.6906


### 100k

In [10]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_cat.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_cat.tail(20000)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
        "class_weight": "balanced", # Adjust weights inversely proportional to class frequencies
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_100k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_100k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_100k_parallel.html")

[I 2026-04-26 22:19:17,160] A new study created in memory with name: no-name-8bab21c8-66a6-45d6-ba8f-27177e5d595a
[I 2026-04-26 22:20:37,367] Trial 3 finished with value: 0.7066377021471911 and parameters: {'max_iter': 707, 'learning_rate': 0.11501016146981184, 'max_depth': 17, 'min_samples_leaf': 443, 'max_features': 0.25639799865259616, 'max_leaf_nodes': 294, 'l2_regularization': 7.13509587088062e-05, 'max_bins': 87}. Best is trial 3 with value: 0.7066377021471911.
[I 2026-04-26 22:20:47,711] Trial 1 finished with value: 0.7026285086003435 and parameters: {'max_iter': 664, 'learning_rate': 0.1137979475990541, 'max_depth': 13, 'min_samples_leaf': 254, 'max_features': 0.5807997224607777, 'max_leaf_nodes': 148, 'l2_regularization': 0.0001339384862473481, 'max_bins': 89}. Best is trial 3 with value: 0.7066377021471911.
[I 2026-04-26 22:21:08,521] Trial 7 finished with value: 0.7089548404440231 and parameters: {'max_iter': 341, 'learning_rate': 0.028289662263978617, 'max_depth': 5, 'min_s


BEST AUC: 0.7116
BEST PARAMETERS:
best_params = {
    "max_iter": 393,
    "learning_rate": 0.028610506052507277,
    "max_depth": 18,
    "min_samples_leaf": 491,
    "max_features": 0.25844075959233276,
    "max_leaf_nodes": 273,
    "l2_regularization": 1.2200390997850423,
    "max_bins": 219,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.4487
  min_samples_leaf    : 0.1912
  max_iter            : 0.1797
  max_leaf_nodes      : 0.0715
  max_depth           : 0.0505
  max_features        : 0.0477
  max_bins            : 0.0106
  l2_regularization   : 0.0002


In [11]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 10)
best_params["learning_rate"] = best_params["learning_rate"] / 10

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingClassifier(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

BEST PARAMS: {'max_iter': 3930, 'learning_rate': 0.0028610506052507275, 'max_depth': 18, 'min_samples_leaf': 491, 'max_features': 0.25844075959233276, 'max_leaf_nodes': 273, 'l2_regularization': 1.2200390997850423, 'max_bins': 219, 'random_state': 42, 'verbose': 0}

Optuna Val AUC:   0.7116
Holdout Test AUC: 0.7145


### Whole training data set

In [22]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights   
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
        "class_weight": "balanced", # Adjust weights inversely proportional to class frequencies
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_full_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_full_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_full_parallel.html")

[I 2026-04-27 08:58:09,083] A new study created in memory with name: no-name-5e1bee10-0ae5-4e7d-acba-4352d129b7b1
[I 2026-04-27 09:00:59,831] Trial 5 finished with value: 0.730651051976217 and parameters: {'max_iter': 715, 'learning_rate': 0.02002511345430492, 'max_depth': 2, 'min_samples_leaf': 447, 'max_features': 0.7262733603117131, 'max_leaf_nodes': 344, 'l2_regularization': 2.382695860880788e-05, 'max_bins': 136}. Best is trial 5 with value: 0.730651051976217.
[I 2026-04-27 09:04:44,442] Trial 8 finished with value: 0.714287509834228 and parameters: {'max_iter': 860, 'learning_rate': 0.0015486556158526503, 'max_depth': 2, 'min_samples_leaf': 225, 'max_features': 0.6177237415706232, 'max_leaf_nodes': 316, 'l2_regularization': 4.038322736508051e-05, 'max_bins': 216}. Best is trial 5 with value: 0.730651051976217.
[I 2026-04-27 09:04:51,455] Trial 0 finished with value: 0.7293961086854731 and parameters: {'max_iter': 397, 'learning_rate': 0.17632308654311252, 'max_depth': 14, 'min_sa


BEST AUC: 0.7365
BEST PARAMETERS:
best_params = {
    "max_iter": 991,
    "learning_rate": 0.009347843785877883,
    "max_depth": 20,
    "min_samples_leaf": 218,
    "max_features": 0.31912233783575017,
    "max_leaf_nodes": 437,
    "l2_regularization": 55.83966409360982,
    "max_bins": 231,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.2721
  max_features        : 0.2092
  max_depth           : 0.1790
  min_samples_leaf    : 0.1326
  max_iter            : 0.0792
  max_leaf_nodes      : 0.0692
  max_bins            : 0.0486
  l2_regularization   : 0.0102


In [23]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 10)
best_params["learning_rate"] = best_params["learning_rate"] / 10

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingClassifier(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

BEST PARAMS: {'max_iter': 9910, 'learning_rate': 0.0009347843785877883, 'max_depth': 20, 'min_samples_leaf': 218, 'max_features': 0.31912233783575017, 'max_leaf_nodes': 437, 'l2_regularization': 55.83966409360982, 'max_bins': 231, 'random_state': 42, 'verbose': 0}

Optuna Val AUC:   0.7365
Holdout Test AUC: 0.7274


## Regression

[Parameters](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html)

### 1k

In [12]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

cols_to_nominal_cat = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5

cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()

ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

ohe_transformer = OneHotEncoder(
    categories=ohe_categories, 
    drop="if_binary", 
    handle_unknown="ignore", 
    sparse_output=False
)

target_transformer_nominal = TargetEncoder(target_type="continuous", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te)
    ],
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

imputed_numeric_preprocessor = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="median")
).set_output(transform="pandas")

gbdt_preprocessor = "passthrough"

Categorical columns:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


In [13]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_reg.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_reg.head(200)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_1k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_1k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_1k_parallel.html")


[I 2026-04-27 01:22:19,178] A new study created in memory with name: no-name-48c63f0b-9d9f-4e49-878a-a104be3e1740
[I 2026-04-27 01:22:22,188] Trial 1 finished with value: 0.3258606371874934 and parameters: {'max_iter': 116, 'learning_rate': 0.06271964316188124, 'max_depth': 7, 'min_samples_leaf': 22, 'max_features': 0.7881571469384545, 'max_leaf_nodes': 3, 'l2_regularization': 0.0071185386843763045, 'max_bins': 171}. Best is trial 1 with value: 0.3258606371874934.
[I 2026-04-27 01:22:24,677] Trial 5 finished with value: 0.33011548807095664 and parameters: {'max_iter': 268, 'learning_rate': 0.1871080879342331, 'max_depth': 3, 'min_samples_leaf': 99, 'max_features': 0.9334980371179025, 'max_leaf_nodes': 3, 'l2_regularization': 0.07811638612032284, 'max_bins': 211}. Best is trial 1 with value: 0.3258606371874934.
[I 2026-04-27 01:22:27,226] Trial 8 finished with value: 0.32721265821484774 and parameters: {'max_iter': 192, 'learning_rate': 0.11537085213297717, 'max_depth': 4, 'min_samples_


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 01:27:05,608] Trial 103 finished with value: 0.320091890168444 and parameters: {'max_iter': 688, 'learning_rate': 0.0053900697558923695, 'max_depth': 2, 'min_samples_leaf': 70, 'max_features': 0.43959482421369184, 'max_leaf_nodes': 7, 'l2_regularization': 4.222970751889853e-05, 'max_bins': 239}. Best is trial 3 with value: 0.316549657384805.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 01:27:07,267] Trial 105 finished with value: 0.31962903584411845 and parameters: {'max_iter': 686, 'learning_rate': 0.005359865547777654, 'max_depth': 2, 'min_samples_leaf': 70, 'max_features': 0.257417126776399, 'max_leaf_nodes': 9, 'l2_regularization': 1.788160109172598e-05, 'max_bins': 213}. Best is trial 3 with value: 0.316549657384805.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 01:27:13,072] Trial 108 finished with value: 0.31934293362950505 and parameters: {'max_iter': 666, 'learning_rate': 0.005426614237589027, 'max_depth': 5, 'min_samples_leaf': 68, 'max_features': 0.2721317322430011, 'max_leaf_nodes': 11, 'l2_regularization': 3.765911242066178e-05, 'max_bins': 212}. Best is trial 3 with value: 0.316549657384805.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 01:27:13,390] Trial 109 finished with value: 0.3199770473204253 and parameters: {'max_iter': 677, 'learning_rate': 0.0060782480436065225, 'max_depth': 5, 'min_samples_leaf': 69, 'max_features': 0.2672739750922162, 'max_leaf_nodes': 9, 'l2_regularization': 1.8511852239514402e-05, 'max_bins': 80}. Best is trial 3 with value: 0.316549657384805.
[I 2026-04-27 01:27:13,413] Trial 110 finished with value: 0.3207890172150961 and parameters: {'max_iter': 685, 'learning_rate': 0.006007995576009099, 'max_depth': 5, 'min_samples_leaf': 70, 'max_features': 0.2650274756791775, 'max_leaf_nodes': 7, 'l2_regularization': 2.0830327433050737e-05, 'max_bins': 63}. Best is trial 3 with value: 0.316549657384805.



[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.3165
BEST PARAMETERS:
best_params = {
    "max_iter": 734,
    "learning_rate": 0.006736563678671684,
    "max_depth": 2,
    "min_samples_leaf": 52,
    "max_features": 0.203174979863919,
    "max_leaf_nodes": 6,
    "l2_regularization": 0.0063066439175882,
    "max_bins": 124,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.4339
  max_iter            : 0.2555
  min_samples_leaf    : 0.1692
  max_bins            : 0.0402
  max_features        : 0.0309
  l2_regularization   : 0.0275
  max_leaf_nodes      : 0.0226
  max_depth           : 0.0202


In [14]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_histgbm.best_params.copy()
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")


final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingRegressor(**best_params)
)

final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print(f"Optuna Val RMSE: {study_histgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")

BEST PARAMS: {'max_iter': 734, 'learning_rate': 0.006736563678671684, 'max_depth': 2, 'min_samples_leaf': 52, 'max_features': 0.203174979863919, 'max_leaf_nodes': 6, 'l2_regularization': 0.0063066439175882, 'max_bins': 124, 'random_state': 42, 'verbose': 0}
Optuna Val RMSE: 0.3165
Holdout Test RMSE: 0.3385


### 10k

In [15]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_reg.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_reg.tail(2000)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_10k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_10k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_10k_parallel.html")

[I 2026-04-27 01:27:14,978] A new study created in memory with name: no-name-88d2f468-3136-482b-9f9c-db79af5fc0e2
[I 2026-04-27 01:27:28,241] Trial 3 finished with value: 0.3035497078288948 and parameters: {'max_iter': 102, 'learning_rate': 0.12581444362441271, 'max_depth': 4, 'min_samples_leaf': 42, 'max_features': 0.9342929149675914, 'max_leaf_nodes': 24, 'l2_regularization': 0.00025799756746807653, 'max_bins': 91}. Best is trial 3 with value: 0.3035497078288948.
[I 2026-04-27 01:27:50,764] Trial 2 finished with value: 0.30617769506845405 and parameters: {'max_iter': 196, 'learning_rate': 0.10945737008125657, 'max_depth': 5, 'min_samples_leaf': 52, 'max_features': 0.9477236649612781, 'max_leaf_nodes': 21, 'l2_regularization': 30.6965302164188, 'max_bins': 172}. Best is trial 3 with value: 0.3035497078288948.
[I 2026-04-27 01:27:52,932] Trial 6 finished with value: 0.3077719372462239 and parameters: {'max_iter': 262, 'learning_rate': 0.094798590048762, 'max_depth': 5, 'min_samples_lea


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 01:59:28,960] Trial 279 finished with value: 0.29795669785341883 and parameters: {'max_iter': 335, 'learning_rate': 0.007687106725908682, 'max_depth': 5, 'min_samples_leaf': 21, 'max_features': 0.43947221529423275, 'max_leaf_nodes': 6, 'l2_regularization': 4.509143142751825e-05, 'max_bins': 70}. Best is trial 177 with value: 0.2974726379147306.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 01:59:31,333] Trial 276 finished with value: 0.2980663005975427 and parameters: {'max_iter': 492, 'learning_rate': 0.007776041965075658, 'max_depth': 5, 'min_samples_leaf': 30, 'max_features': 0.5137140671323854, 'max_leaf_nodes': 6, 'l2_regularization': 1.035969494989601e-05, 'max_bins': 71}. Best is trial 177 with value: 0.2974726379147306.
[I 2026-04-27 01:59:37,215] Trial 281 finished with value: 0.29800477583467566 and parameters: {'max_iter': 440, 'learning_rate': 0.007660206716520857, 'max_depth': 5, 'min_samples_leaf': 29, 'max_features': 0.4284943848641099, 'max_leaf_nodes': 6, 'l2_regularization': 2.0612246315627093e-05, 'max_bins': 67}. Best is trial 177 with value: 0.2974726379147306.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 01:59:41,846] Trial 282 finished with value: 0.2976936244965801 and parameters: {'max_iter': 440, 'learning_rate': 0.007621504565986444, 'max_depth': 8, 'min_samples_leaf': 22, 'max_features': 0.4342311348978581, 'max_leaf_nodes': 6, 'l2_regularization': 1.912930046479809e-05, 'max_bins': 67}. Best is trial 177 with value: 0.2974726379147306.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 01:59:42,249] Trial 283 finished with value: 0.297888221030669 and parameters: {'max_iter': 448, 'learning_rate': 0.009683613525138817, 'max_depth': 5, 'min_samples_leaf': 22, 'max_features': 0.43791417291074924, 'max_leaf_nodes': 6, 'l2_regularization': 4.8815470393993206e-05, 'max_bins': 66}. Best is trial 177 with value: 0.2974726379147306.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.2975
BEST PARAMETERS:
best_params = {
    "max_iter": 439,
    "learning_rate": 0.009812688905374576,
    "max_depth": 5,
    "min_samples_leaf": 23,
    "max_features": 0.3270420473977633,
    "max_leaf_nodes": 5,
    "l2_regularization": 0.0003377040719829344,
    "max_bins": 71,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.7566
  max_iter            : 0.1402
  max_features        : 0.0379
  max_depth           : 0.0345
  max_bins            : 0.0131
  max_leaf_nodes      : 0.0128
  min_samples_leaf    : 0.0047
  l2_regularization   : 0.0002


In [16]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 2)
best_params["learning_rate"] = best_params["learning_rate"] / 2

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingRegressor(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)

BEST PARAMS: {'max_iter': 878, 'learning_rate': 0.004906344452687288, 'max_depth': 5, 'min_samples_leaf': 23, 'max_features': 0.3270420473977633, 'max_leaf_nodes': 5, 'l2_regularization': 0.0003377040719829344, 'max_bins': 71, 'random_state': 42, 'verbose': 0}

Optuna Val RMSE:   0.2975
Holdout Test RMSE: 0.3137


### 100k

In [17]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_reg.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_reg.tail(20000)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_100k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_100k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_100k_parallel.html")

[I 2026-04-27 01:59:47,531] A new study created in memory with name: no-name-c7d2809e-4b8a-42c4-b67b-bbb3faddff21
[I 2026-04-27 02:00:07,657] Trial 0 finished with value: 0.3008679946830622 and parameters: {'max_iter': 721, 'learning_rate': 0.1817332207250368, 'max_depth': 6, 'min_samples_leaf': 252, 'max_features': 0.6551818677209466, 'max_leaf_nodes': 207, 'l2_regularization': 3.392150229324541, 'max_bins': 147}. Best is trial 0 with value: 0.3008679946830622.
[I 2026-04-27 02:00:35,712] Trial 6 finished with value: 0.2998249904709847 and parameters: {'max_iter': 563, 'learning_rate': 0.03618011117970041, 'max_depth': 4, 'min_samples_leaf': 348, 'max_features': 0.5420069448914628, 'max_leaf_nodes': 239, 'l2_regularization': 0.005205100230608828, 'max_bins': 122}. Best is trial 6 with value: 0.2998249904709847.
[I 2026-04-27 02:00:54,953] Trial 4 finished with value: 0.3001042738158214 and parameters: {'max_iter': 196, 'learning_rate': 0.07406041933928344, 'max_depth': 20, 'min_sample


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 04:08:44,325] Trial 280 finished with value: 0.29975838627981505 and parameters: {'max_iter': 486, 'learning_rate': 0.023302956365122467, 'max_depth': 8, 'min_samples_leaf': 130, 'max_features': 0.4318004784246775, 'max_leaf_nodes': 16, 'l2_regularization': 4.730411619054606e-05, 'max_bins': 141}. Best is trial 174 with value: 0.29946656775009506.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 04:08:58,863] Trial 278 finished with value: 0.29971911069669394 and parameters: {'max_iter': 530, 'learning_rate': 0.022713910865204964, 'max_depth': 8, 'min_samples_leaf': 479, 'max_features': 0.4341144530562272, 'max_leaf_nodes': 399, 'l2_regularization': 0.0989331024324619, 'max_bins': 103}. Best is trial 174 with value: 0.29946656775009506.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 04:09:13,044] Trial 276 finished with value: 0.2999311941643946 and parameters: {'max_iter': 522, 'learning_rate': 0.023453892357916808, 'max_depth': 20, 'min_samples_leaf': 480, 'max_features': 0.36450057815705433, 'max_leaf_nodes': 276, 'l2_regularization': 4.63741317964026e-05, 'max_bins': 164}. Best is trial 174 with value: 0.29946656775009506.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 04:09:14,421] Trial 281 finished with value: 0.2997588511492623 and parameters: {'max_iter': 486, 'learning_rate': 0.02329381583538688, 'max_depth': 8, 'min_samples_leaf': 478, 'max_features': 0.4382039419105225, 'max_leaf_nodes': 254, 'l2_regularization': 4.590552470183276e-05, 'max_bins': 142}. Best is trial 174 with value: 0.29946656775009506.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.2995
BEST PARAMETERS:
best_params = {
    "max_iter": 480,
    "learning_rate": 0.014304115664264032,
    "max_depth": 8,
    "min_samples_leaf": 259,
    "max_features": 0.4398281267806546,
    "max_leaf_nodes": 43,
    "l2_regularization": 0.00024044149018689098,
    "max_bins": 147,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.4199
  min_samples_leaf    : 0.2854
  max_iter            : 0.1618
  max_depth           : 0.0753
  max_features        : 0.0395
  max_leaf_nodes      : 0.0119
  max_bins            : 0.0059
  l2_regularization   : 0.0002


In [18]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 10)
best_params["learning_rate"] = best_params["learning_rate"] / 10

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingRegressor(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)

BEST PARAMS: {'max_iter': 4800, 'learning_rate': 0.001430411566426403, 'max_depth': 8, 'min_samples_leaf': 259, 'max_features': 0.4398281267806546, 'max_leaf_nodes': 43, 'l2_regularization': 0.00024044149018689098, 'max_bins': 147, 'random_state': 42, 'verbose': 0}

Optuna Val RMSE:   0.2995
Holdout Test RMSE: 0.2989


### Whole training data set

In [19]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_reg[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_reg[split_index:]

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights   
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        cv_scores.append(np.sqrt(mean_squared_error(y_val, preds)))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_full_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_full_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_full_parallel.html")

[I 2026-04-27 04:10:17,878] A new study created in memory with name: no-name-8071a024-2afd-4218-aca6-23aa1c728e3c
[I 2026-04-27 04:12:01,501] Trial 3 finished with value: 0.2335040479748809 and parameters: {'max_iter': 560, 'learning_rate': 0.1538926722247836, 'max_depth': 9, 'min_samples_leaf': 166, 'max_features': 0.9953245924947658, 'max_leaf_nodes': 20, 'l2_regularization': 14.911734213250796, 'max_bins': 113}. Best is trial 3 with value: 0.2335040479748809.
[I 2026-04-27 04:15:13,861] Trial 1 finished with value: 0.23452134072696754 and parameters: {'max_iter': 349, 'learning_rate': 0.007267650762639977, 'max_depth': 5, 'min_samples_leaf': 74, 'max_features': 0.6854337362666485, 'max_leaf_nodes': 80, 'l2_regularization': 0.0033974165775483346, 'max_bins': 99}. Best is trial 3 with value: 0.2335040479748809.
[I 2026-04-27 04:17:16,509] Trial 6 finished with value: 0.23463948903729714 and parameters: {'max_iter': 467, 'learning_rate': 0.15350378043008286, 'max_depth': 14, 'min_sampl


BEST RMSE: 0.2327
BEST PARAMETERS:
best_params = {
    "max_iter": 679,
    "learning_rate": 0.018532518685825526,
    "max_depth": 15,
    "min_samples_leaf": 362,
    "max_features": 0.4243152639220175,
    "max_leaf_nodes": 206,
    "l2_regularization": 0.06879760344829619,
    "max_bins": 213,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.4641
  min_samples_leaf    : 0.2052
  max_depth           : 0.1408
  max_iter            : 0.0698
  max_leaf_nodes      : 0.0574
  max_bins            : 0.0374
  max_features        : 0.0249
  l2_regularization   : 0.0005


In [20]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 10)
best_params["learning_rate"] = best_params["learning_rate"] / 10

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingRegressor(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)

BEST PARAMS: {'max_iter': 6790, 'learning_rate': 0.0018532518685825527, 'max_depth': 15, 'min_samples_leaf': 362, 'max_features': 0.4243152639220175, 'max_leaf_nodes': 206, 'l2_regularization': 0.06879760344829619, 'max_bins': 213, 'random_state': 42, 'verbose': 0}

Optuna Val RMSE:   0.2327
Holdout Test RMSE: 0.2838
